# 01 — NumPy cơ bản cho Deep Learning

Mục tiêu: làm quen với **tensor (mảng nhiều chiều)** và 3 phép toán sẽ xuất hiện ở khắp nơi trong GPT:
1. Nhân ma trận
2. Softmax
3. Cross-entropy

> Quy tắc: chạy từng ô, **đoán kết quả trước khi chạy**, rồi so sánh.

In [ ]:
import numpy as np
np.random.seed(0)

## 1. Vector, ma trận và `shape`

Trong deep learning, 80% lỗi là lỗi **shape**. Hãy luôn in `shape` ra.

In [ ]:
x = np.array([1.0, 2.0, 3.0])          # vector, shape (3,)
W = np.random.randn(4, 3)              # ma trận 4x3
print("x.shape =", x.shape)
print("W.shape =", W.shape)

## 2. Nhân ma trận — phép toán cốt lõi của mạng nơ-ron

Một lớp nơ-ron tuyến tính chỉ là: `y = W @ x + b`.

Mỗi phần tử `y[i]` = tích vô hướng (dot product) giữa hàng `i` của `W` và `x`.

In [ ]:
b = np.zeros(4)
y = W @ x + b
print("y.shape =", y.shape)

# Tự kiểm tra: tính y[0] bằng vòng lặp thủ công
y0_manual = sum(W[0, j] * x[j] for j in range(3))
print(y[0], y0_manual, np.isclose(y[0], y0_manual))

### ✏️ Bài tập 1
Viết hàm `matmul(A, B)` bằng **3 vòng lặp for** (không dùng `@`), so sánh kết quả với `A @ B`.

In [ ]:
def matmul(A, B):
    # TODO: A có shape (n, k), B có shape (k, m) -> kết quả (n, m)
    raise NotImplementedError

# A = np.random.randn(2, 3); B = np.random.randn(3, 4)
# print(np.allclose(matmul(A, B), A @ B))

## 3. Broadcasting và batch

Thực tế ta xử lý nhiều mẫu cùng lúc (một **batch**). Với `X` shape `(batch, 3)`:
`Y = X @ W.T + b` → shape `(batch, 4)`. `b` shape `(4,)` được tự động "broadcast" cho mọi hàng.

In [ ]:
X = np.random.randn(5, 3)      # 5 mẫu, mỗi mẫu 3 đặc trưng
Y = X @ W.T + b
print(Y.shape)

## 4. Softmax — biến điểm số thành xác suất

$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

GPT dùng softmax ở 2 nơi: tính **attention weights** và tính **xác suất token tiếp theo**.

In [ ]:
def softmax_naive(z):
    e = np.exp(z)
    return e / e.sum()

z = np.array([2.0, 1.0, 0.1])
p = softmax_naive(z)
print(p, p.sum())

# Vấn đề: số lớn gây tràn số
print(softmax_naive(np.array([1000.0, 1001.0])))

### ✏️ Bài tập 2
Sửa softmax cho **ổn định số học**: trừ đi `max(z)` trước khi lấy `exp`.
Vì sao trừ một hằng số không làm thay đổi kết quả? (Viết lời giải thích vào `notes/softmax.md`.)
Hàm phải hoạt động cho cả input 2 chiều `(batch, classes)` — softmax theo trục cuối.

In [ ]:
def softmax(z):
    # TODO
    raise NotImplementedError

# print(softmax(np.array([1000.0, 1001.0])))          # mong đợi ~[0.269, 0.731]
# print(softmax(np.random.randn(4, 5)).sum(axis=-1))   # mong đợi toàn 1.0

## 5. Cross-entropy — hàm mất mát của mọi LLM

Nếu mô hình gán xác suất `p` cho đáp án đúng, loss = $-\log p$.

- Đoán đúng tự tin (p ≈ 1) → loss ≈ 0
- Đoán sai tự tin (p ≈ 0) → loss rất lớn

**GPT được huấn luyện bằng đúng hàm này**: tối thiểu hóa $-\log P(\text{token đúng tiếp theo})$.

In [ ]:
for prob in [0.99, 0.5, 0.1, 0.001]:
    print(f"p = {prob:<6} -> loss = {-np.log(prob):.3f}")

### ✏️ Bài tập 3
Viết `cross_entropy(logits, targets)`:
- `logits`: shape `(batch, num_classes)` — điểm số thô chưa qua softmax
- `targets`: shape `(batch,)` — chỉ số lớp đúng
- Trả về loss **trung bình** trên batch.

Câu hỏi suy nghĩ: với 50 000 token trong từ điển và mô hình đoán ngẫu nhiên đều, loss ban đầu là bao nhiêu?
(Gợi ý: $-\log(1/50000)$. Con số này sẽ gặp lại khi bạn huấn luyện GPT!)

In [ ]:
def cross_entropy(logits, targets):
    # TODO
    raise NotImplementedError

# logits = np.array([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3]])
# targets = np.array([0, 1])
# print(cross_entropy(logits, targets))   # mong đợi ~0.3185